# Notebook 2 — Visualisation des destinations Kayak

**Projet :** Plan your trip with Kayak — Bloc 1 Jedha CDSD
**Auteur :** Aymeric Lahonde

## Objectif

Une fois la collecte météo + hôtels terminée (Notebook 1), on charge les CSV finaux et on produit les **2 cartes interactives** demandées par Kayak :
- **Carte 1** : Top 5 destinations selon le score météo
- **Carte 2** : Top 20 hôtels dans ces 5 destinations

On lit les données directement depuis les CSV en local. La même logique pourrait lire depuis la base RDS (cf. notebook 1, partie ETL).


In [ ]:
# Imports
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print("Imports OK")


---
## 1. Chargement des données

On a deux CSV à lire :
- `weather_cities.csv` — la météo agrégée par ville + le score météo (1 ligne par ville)
- `hotels_booking.csv` — les hôtels scrapés sur Booking.com

Les chemins sont relatifs au notebook (qui est dans `notebooks/`), donc `../data/`.


In [ ]:
# Lecture des données météo
df_weather = pd.read_csv("../data/weather_cities.csv")
df_weather = df_weather.sort_values("rank")

print("Shape :", df_weather.shape)
df_weather.head()


In [ ]:
# Petit coup d'œil sur les colonnes et les types
df_weather.info()


In [ ]:
# Statistiques descriptives sur les variables numériques
df_weather[["temp_max_mean", "pop_mean", "rain_total", "humidity_mean", "weather_score"]].describe().round(2)


In [ ]:
# Lecture des hôtels
df_hotels = pd.read_csv("../data/hotels_booking.csv")

print("Shape :", df_hotels.shape)
df_hotels.head()


In [ ]:
# On regarde combien d'hôtels par ville
df_hotels["city"].value_counts().head(10)


---
## 2. Top 5 des destinations selon le score météo

Le score est calculé dans le Notebook 1 (combinaison température, probabilité de pluie, humidité, etc.). Plus il est haut, mieux c'est.


In [ ]:
# On garde les 5 meilleures villes
top5 = df_weather[df_weather["rank"] <= 5].copy()

# Affichage propre
top5[["rank", "city", "temp_max_mean", "pop_mean", "rain_total", "weather_score"]]


---
## 3. Carte 1 — Top 5 destinations

On utilise **Plotly Express** pour avoir une carte interactive (zoom, hover sur les villes). Toutes les 35 villes sont affichées avec une taille et une couleur proportionnelles au score météo, et on rajoute des étoiles dorées sur les 5 meilleures.


In [ ]:
# Carte de base avec les 35 villes
fig_destinations = px.scatter_map(
    df_weather,
    lat="lat",
    lon="lon",
    color="weather_score",
    size="weather_score",
    hover_name="city",
    hover_data={
        "weather_score": ":.1f",
        "temp_max_mean": ":.1f",
        "pop_mean": ":.1%",
        "rain_total": ":.1f",
        "rank": True,
    },
    color_continuous_scale="RdYlGn",
    size_max=30,
    zoom=4.5,
    center={"lat": 46.5, "lon": 2.5},
    map_style="open-street-map",
    title="Meilleures destinations en France selon la météo (5 jours)",
)

# On ajoute des étoiles dorées sur le Top 5
fig_destinations.add_trace(
    go.Scattermap(
        lat=top5["lat"],
        lon=top5["lon"],
        mode="markers+text",
        text=top5["rank"].astype(str) + ". " + top5["city"],
        textposition="top right",
        marker=dict(size=22, color="gold", symbol="star"),
        name="Top 5",
        hoverinfo="skip",
    )
)

fig_destinations.update_layout(height=650, margin={"r": 0, "t": 60, "l": 0, "b": 0})
fig_destinations.show()


---
## 4. Sélection des 20 meilleurs hôtels dans le Top 5

On filtre les hôtels :
1. Uniquement ceux situés dans une des 5 meilleures villes météo
2. On enlève les hôtels sans score Booking (NaN)
3. On trie par score décroissant et on garde les 20 premiers


In [ ]:
# Liste des 5 meilleures villes
top5_cities = top5["city"].tolist()
print("Villes du Top 5 :", top5_cities)


In [ ]:
# Filtrage des hôtels
df_hotels_top = (
    df_hotels[df_hotels["city"].isin(top5_cities)]
    .dropna(subset=["score"])
    .sort_values("score", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# On ajoute un classement
df_hotels_top["hotel_rank"] = df_hotels_top.index + 1

print(f"{len(df_hotels_top)} hôtels sélectionnés")
df_hotels_top[["hotel_rank", "hotel_name", "city", "score"]].head(10)


---
## 5. Carte 2 — Top 20 hôtels

Même principe que la carte 1, en zoomant sur les hôtels et en utilisant une palette bleue pour différencier des villes.


In [ ]:
fig_hotels = px.scatter_map(
    df_hotels_top,
    lat="lat",
    lon="lon",
    color="score",
    size="score",
    hover_name="hotel_name",
    hover_data={"city": True, "score": ":.1f", "hotel_rank": True},
    color_continuous_scale="Blues",
    size_max=25,
    zoom=4.5,
    center={"lat": 44.5, "lon": 5.0},
    map_style="open-street-map",
    title="Top 20 hôtels dans les 5 meilleures destinations",
)

# Numéros de classement sur la carte
fig_hotels.add_trace(
    go.Scattermap(
        lat=df_hotels_top["lat"],
        lon=df_hotels_top["lon"],
        mode="text",
        text=df_hotels_top["hotel_rank"].astype(str),
        textfont=dict(size=10, color="white"),
        hoverinfo="skip",
        showlegend=False,
    )
)

fig_hotels.update_layout(height=650, margin={"r": 0, "t": 60, "l": 0, "b": 0})
fig_hotels.show()


---
## 6. Export des cartes en HTML

Les cartes Plotly peuvent être sauvegardées en HTML standalone — utile pour les partager ou les insérer dans une présentation.


In [ ]:
import os

out_dir = "../outputs/maps"
os.makedirs(out_dir, exist_ok=True)

fig_destinations.write_html(f"{out_dir}/map_destinations.html")
fig_hotels.write_html(f"{out_dir}/map_hotels.html")

print("Cartes exportées dans outputs/maps/")


---
## Conclusion

On a produit les **2 livrables visuels** demandés par Kayak :
- Une carte des 35 villes françaises avec mise en avant des 5 meilleures (météo)
- Une carte des 20 meilleurs hôtels dans ces 5 villes (score Booking)

**Top 5 destinations (score sur 100, sur 5 jours) :**

| # | Ville | Score |
|---|-------|-------|
| 1 | Aix-en-Provence | 88,9 |
| 2 | Avignon | 87,8 |
| 3 | Marseille | 86,7 |
| 4 | Paris | 85,5 |
| 5 | Grenoble | 82,3 |

**Pistes d'amélioration :**
- Pondérer le score météo par la saison (été vs hiver)
- Croiser avec le prix moyen des hôtels (donnée à scraper)
- Lecture directe depuis RDS pour avoir les données toujours à jour
